# Binary Classification Model

### import libraries

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

### load cleaned dataset

In [2]:
df = pd.read_csv("data/cleaned_cafe_sales.csv")
df

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2.0,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4.0,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4.0,1.0,8.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2.0,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2.0,2.0,4.0,Digital Wallet,In-store,2023-06-11
...,...,...,...,...,...,...,...,...
9995,TXN_7672686,Coffee,2.0,2.0,4.0,Payment Method,UNKNOWN,2023-08-30
9996,TXN_9659401,Juice,3.0,3.0,3.0,Digital Wallet,Takeaway,2023-06-02
9997,TXN_5255387,Coffee,4.0,2.0,8.0,Digital Wallet,Takeaway,2023-03-02
9998,TXN_7695629,Cookie,3.0,3.0,3.0,Digital Wallet,Takeaway,2023-12-02


### check column names

In [3]:
df.columns

Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='object')

### check datatype

In [4]:
df.dtypes

Transaction ID       object
Item                 object
Quantity            float64
Price Per Unit      float64
Total Spent         float64
Payment Method       object
Location             object
Transaction Date     object
dtype: object

In [5]:
df.shape

(10000, 8)

### check null values/duplicates

In [6]:
df.isnull().sum()

Transaction ID      0
Item                0
Quantity            0
Price Per Unit      0
Total Spent         0
Payment Method      0
Location            0
Transaction Date    0
dtype: int64

### check statistical summary

In [7]:
df.describe()

,Quantity,Price Per Unit,Total Spent
count,10000.000000,10000.00000,10000.00000
mean,3.027100,2.95265,8.87795
std,1.384614,1.24396,5.86059
min,1.000000,1.00000,1.00000
25%,2.000000,2.00000,4.00000
50%,3.000000,3.00000,8.00000
75%,4.000000,4.00000,12.00000
max,5.000000,5.00000,25.00000


### create binary target variable

In [8]:
df["High_Quantity"] = (df["Quantity"] >= 3).astype(int)

### count target variable

In [9]:
df["High_Quantity"].value_counts()

High_Quantity
1    6204
0    3796
Name: count, dtype: int64

### check target variable percentage distribution

In [10]:
df["High_Quantity"].value_counts(normalize=True) * 100

High_Quantity
1    62.04
0    37.96
Name: proportion, dtype: float64

### seperate features and target columns

In [11]:
X = df[
    [
        "Item",
        "Price Per Unit",
        "Payment Method",
        "Location",
        "Transaction Date"
    ]
]
y = df["High_Quantity"]

In [12]:
## check
print("Features:")
print(X.head())
print("\nTarget:")
print(y.head())

Features:
     Item  Price Per Unit  Payment Method  Location Transaction Date
0  Coffee             2.0     Credit Card  Takeaway       2023-09-08
1    Cake             3.0            Cash  In-store       2023-05-16
2  Cookie             1.0     Credit Card  In-store       2023-07-19
3   Salad             5.0         UNKNOWN   UNKNOWN       2023-04-27
4  Coffee             2.0  Digital Wallet  In-store       2023-06-11

Target:
0    0
1    1
2    1
3    0
4    0
Name: High_Quantity, dtype: int64


### Transaction Date Processing (Feature Engineering)

In [13]:
# Convert Transaction Date to datetime with error handling
X["Transaction Date"] = pd.to_datetime(X["Transaction Date"], format='mixed', errors='coerce')

C:\Users\prime\AppData\Local\Temp\ipykernel_23176\4127503831.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Transaction Date"] = pd.to_datetime(X["Transaction Date"], format='mixed', errors='coerce')


In [14]:
# Extract useful date features
X["Transaction_Year"] = X["Transaction Date"].dt.year
X["Transaction_Month"] = X["Transaction Date"].dt.month
X["Transaction_Day"] = X["Transaction Date"].dt.day
X["Transaction_DayOfWeek"] = X["Transaction Date"].dt.dayofweek

C:\Users\prime\AppData\Local\Temp\ipykernel_23176\1039986480.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Transaction_Year"] = X["Transaction Date"].dt.year
C:\Users\prime\AppData\Local\Temp\ipykernel_23176\1039986480.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X["Transaction_Month"] = X["Transaction Date"].dt.month
C:\Users\prime\AppData\Local\Temp\ipykernel_23176\1039986480.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[ro

In [15]:
# Drop original date column
X = X.drop(columns=["Transaction Date"])

In [16]:
X.head()

,Item,Price Per Unit,Payment Method,Location,Transaction_Year,Transaction_Month,Transaction_Day,Transaction_DayOfWeek
0,Coffee,2.0,Credit Card,Takeaway,2023.0,9.0,8.0,4.0
1,Cake,3.0,Cash,In-store,2023.0,5.0,16.0,1.0
2,Cookie,1.0,Credit Card,In-store,2023.0,7.0,19.0,2.0
3,Salad,5.0,UNKNOWN,UNKNOWN,2023.0,4.0,27.0,3.0
4,Coffee,2.0,Digital Wallet,In-store,2023.0,6.0,11.0,6.0


### Categorical Columns Encoding

In [17]:
## Apply One-Hot Encoding
X = pd.get_dummies(X, drop_first=True)
X.head()

,Price Per Unit,Transaction_Year,Transaction_Month,Transaction_Day,Transaction_DayOfWeek,Item_Coffee,Item_Cookie,Item_ERROR,Item_Juice,Item_Salad,...,Item_Tea,Item_UNKNOWN,Payment Method_Credit Card,Payment Method_Digital Wallet,Payment Method_ERROR,Payment Method_Payment Method,Payment Method_UNKNOWN,Location_In-store,Location_Takeaway,Location_UNKNOWN
0,2.0,2023.0,9.0,8.0,4.0,True,False,False,False,False,...,False,False,True,False,False,False,False,False,True,False
1,3.0,2023.0,5.0,16.0,1.0,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,1.0,2023.0,7.0,19.0,2.0,False,True,False,False,False,...,False,False,True,False,False,False,False,True,False,False
3,5.0,2023.0,4.0,27.0,3.0,False,False,False,False,True,...,False,False,False,False,False,False,True,False,False,True
4,2.0,2023.0,6.0,11.0,6.0,True,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False


In [18]:
## Recheck datatypes
print(X.dtypes)

Price Per Unit                   float64
Transaction_Year                 float64
Transaction_Month                float64
Transaction_Day                  float64
Transaction_DayOfWeek            float64
Item_Coffee                         bool
Item_Cookie                         bool
Item_ERROR                          bool
Item_Juice                          bool
Item_Salad                          bool
Item_Sandwich                       bool
Item_Smoothie                       bool
Item_Tea                            bool
Item_UNKNOWN                        bool
Payment Method_Credit Card          bool
Payment Method_Digital Wallet       bool
Payment Method_ERROR                bool
Payment Method_Payment Method       bool
Payment Method_UNKNOWN              bool
Location_In-store                   bool
Location_Takeaway                   bool
Location_UNKNOWN                    bool
dtype: object


### check missing values in input features and target colulmns

In [19]:
print("Missing values:")
print(X.isnull().sum().sum())

Missing values:
1840


In [20]:
print("Target missing values:", y.isnull().sum())

Target missing values: 0


In [21]:
## check missing value of every column
print(X.isnull().sum())

Price Per Unit                     0
Transaction_Year                 460
Transaction_Month                460
Transaction_Day                  460
Transaction_DayOfWeek            460
Item_Coffee                        0
Item_Cookie                        0
Item_ERROR                         0
Item_Juice                         0
Item_Salad                         0
Item_Sandwich                      0
Item_Smoothie                      0
Item_Tea                           0
Item_UNKNOWN                       0
Payment Method_Credit Card         0
Payment Method_Digital Wallet      0
Payment Method_ERROR               0
Payment Method_Payment Method      0
Payment Method_UNKNOWN             0
Location_In-store                  0
Location_Takeaway                  0
Location_UNKNOWN                   0
dtype: int64


### Drop rows with missing date values from both X and y

In [22]:
valid_rows = X.dropna().index
X = X.loc[valid_rows]
y = y.loc[valid_rows]

### Recheck missing values

In [23]:
print("Remaining missing values in X:", X.isnull().sum().sum())
print("X shape:", X.shape)
print("y shape:", y.shape)

Remaining missing values in X: 0
X shape: (9540, 22)
y shape: (9540,)


### Splitting the dataset into 80% training data (`X_train`, `y_train`) and 20% testing data (`X_test`, `y_test`) using `train_test_split`.

In [24]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, 
    random_state=42
)

In [25]:
# check shapes
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (7632, 22)
Testing data: (1908, 22)


### Training a Logistic Regression model on the preprocessed training dataset

In [26]:
from sklearn.linear_model import LogisticRegression
# Create the Logistic Regression model
model = LogisticRegression(max_iter=1000)
# Train the model
model.fit(X_train, y_train)
print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


C:\Users\prime\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Make predictions on the test data

In [27]:
predictions = model.predict(X_test)
print("Predictions generated successfully!")
print(predictions[:10])

Predictions generated successfully!
[1 1 1 1 1 1 1 1 1 1]


### Evaluating model performance on the test dataset (`y_test` vs `predictions`) using key classification metrics: Accuracy, Precision, Recall, and F1-Score.

In [28]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
# Calculate evaluation metrics
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, average="weighted")
recall = recall_score(y_test, predictions, average="weighted")
f1 = f1_score(y_test, predictions, average="weighted")



C:\Users\prime\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [29]:
# Display results
print("Model Evaluation Results")
print("------------------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-Score :", round(f1, 4))

Model Evaluation Results
------------------------
Accuracy : 0.6158
Precision: 0.3792
Recall   : 0.6158
F1-Score : 0.4694


### check classification report

In [30]:
from sklearn.metrics import classification_report
print("Classification Report")
print("---------------------")
print(classification_report(y_test, predictions))

Classification Report
---------------------
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       733
           1       0.62      1.00      0.76      1175

    accuracy                           0.62      1908
   macro avg       0.31      0.50      0.38      1908
weighted avg       0.38      0.62      0.47      1908



C:\Users\prime\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\prime\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\prime\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

### Training an alternative non-linear algorithm using `DecisionTreeClassifier` to compare performance against Logistic Regression.

In [31]:
from sklearn.tree import DecisionTreeClassifier
# Create Decision Tree model
tree_model = DecisionTreeClassifier(
    random_state=42
)
# Train the model
tree_model.fit(X_train, y_train)
print("Decision Tree model trained successfully!")

Decision Tree model trained successfully!


In [32]:
# Make predictions using Decision Tree
tree_predictions = tree_model.predict(X_test)
print("Decision Tree predictions generated successfully!")

Decision Tree predictions generated successfully!


In [33]:
# Decision Tree evaluation metrics
tree_accuracy = accuracy_score(y_test, tree_predictions)
tree_precision = precision_score(y_test, tree_predictions, average="weighted")
tree_recall = recall_score(y_test, tree_predictions, average="weighted")
tree_f1 = f1_score(y_test, tree_predictions, average="weighted")
print("Decision Tree Evaluation Results")
print("--------------------------------")
print("Accuracy :", round(tree_accuracy, 4))
print("Precision:", round(tree_precision, 4))
print("Recall   :", round(tree_recall, 4))
print("F1-Score :", round(tree_f1, 4))

Decision Tree Evaluation Results
--------------------------------
Accuracy : 0.5131
Precision: 0.5177
Recall   : 0.5131
F1-Score : 0.5152


### Decission Tree Classification Report

In [34]:
print("Decision Tree Classification Report")
print("---------------------------------")
print(classification_report(y_test, tree_predictions))

Decision Tree Classification Report
---------------------------------
              precision    recall  f1-score   support

           0       0.37      0.39      0.38       733
           1       0.61      0.59      0.60      1175

    accuracy                           0.51      1908
   macro avg       0.49      0.49      0.49      1908
weighted avg       0.52      0.51      0.52      1908



### compare Both Models

In [35]:
comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree"],
    "Accuracy": [accuracy, tree_accuracy],
    "Precision": [precision, tree_precision],
    "Recall": [recall, tree_recall],
    "F1-Score": [f1, tree_f1]
})

comparison

,Model,Accuracy,Precision,Recall,F1-Score
0,Logistic Regression,0.615828,0.379244,0.615828,0.469412
1,Decision Tree,0.513103,0.517667,0.513103,0.515201


## ## Model Performance Summary  
The Logistic Regression model achieved an accuracy of 61.58%, while the Decision Tree achieved an accuracy of 51.00%. Logistic Regression performed better overall based on accuracy and weighted recall, although its classification report shows that it struggled to identify the low-quantity class. The Decision Tree provided more balanced predictions between the two classes but had lower overall accuracy. The model could be improved by addressing class imbalance, creating more informative features, tuning model hyperparameters, and testing additional classification algorithms.